# SEC Company Status

In [1]:
%%javascript
IPython.OutputArea.auto_scroll_threshold = 9999;

<IPython.core.display.Javascript object>

In [2]:
# ----// HEADER // -------------------------------------------------------------
# ------------------------------------------------------------------------------
# PROJECT NAME: 'Zeus' (Contains Projects 'Apollo', 'Artemis', and 'Athena')
# AUTHOR:       Tonmoy Roy
# CONTACT:      tonmoyroy@duck.com
# LICENSE:      GNU LGPLv3
# ------------------------------------------------------------------------------

In [3]:
# IMPORT LIBRARIES
import pandas as pd
import plotly.graph_objects as go
import requests

from plotly.subplots import make_subplots
from time import sleep

# ADJUST CONFIGURATOINS
pd.set_option("display.max_columns", None)

# Ticker Symbol Input

In [4]:
# SELECT ASSET TICKER SYMBOL
__tickSymb = 'TSLA'

In [5]:
# GENERATE REQUEST HEADER
requHead = {'User-Agent': "tonmoyroy@duck.com"}
# GET CIK JSON RESPONSE AND PRE-PROCESS
requResp = requests.get("https://www.sec.gov/files/company_tickers.json", headers = requHead)
requResp = pd.json_normalize(pd.json_normalize(requResp.json(), max_level = 0).values[0])
requResp["cik_str"] = requResp["cik_str"].astype(str).str.zfill(10)
requResp.set_index('ticker', inplace = True)

In [6]:
# GET A LIST OF AVAILABLE PARAMETERS
for indx in range(len(requResp)):
    if str(requResp.index[indx]) == __tickSymb:
        # DERIVE CIK VALUE
        cikValue = str(requResp['cik_str'].iloc[indx])
        # DERIVE A LIST OF AVAILABLE PARAMETERS
        try:
            paraList = (requests.get("https://data.sec.gov/api/xbrl/companyfacts/CIK"+ cikValue + ".json", headers = requHead)).json()['facts']['us-gaap'].keys()
            paraList = list(paraList)
        except Exception:
            paraList = (requests.get("https://data.sec.gov/api/xbrl/companyfacts/CIK"+ cikValue + ".json", headers = requHead)).json()['facts']['ifrs-full'].keys()
            paraList = list(paraList)
        break
# SORT LIST
paraList.sort()
# PRINT LIST
print('\n'.join(map(str, paraList)))

AccountsAndNotesReceivableNet
AccountsPayableCurrent
AccountsReceivableNetCurrent
AccretionAmortizationOfDiscountsAndPremiumsInvestments
AccrualForEnvironmentalLossContingencies
AccrualForEnvironmentalLossContingenciesPayments
AccruedEnvironmentalLossContingenciesCurrent
AccruedEnvironmentalLossContingenciesNoncurrent
AccruedLiabilitiesCurrent
AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment
AccumulatedOtherComprehensiveIncomeLossNetOfTax
AdditionalPaidInCapital
AdditionalPaidInCapitalCommonStock
AdjustmentsNoncashItemsToReconcileNetIncomeLossToCashProvidedByUsedInOperatingActivities
AdjustmentsToAdditionalPaidInCapitalEquityComponentOfConvertibleDebt
AdjustmentsToAdditionalPaidInCapitalOther
AdjustmentsToAdditionalPaidInCapitalSharebasedCompensationRequisiteServicePeriodRecognitionValue
AdjustmentsToAdditionalPaidInCapitalTaxEffectFromShareBasedCompensation
AdjustmentsToAdditionalPaidInCapitalWarrantIssued
AdvertisingRevenueCost
AllocatedShareBasedCompensationE

In [7]:
def requestData(__cikValue__, __accoType__, __variName__, __requHead__ = requHead):
    if str(__accoType__) == 'gaap':
        return requests.get('https://data.sec.gov/api/xbrl/companyconcept/' + 'CIK' + __cikValue__ + '/us-gaap/' + __variName__ + '.json', headers = __requHead__)
    elif str(__accoType__) == 'ifrs':
        return requests.get('https://data.sec.gov/api/xbrl/companyconcept/' + 'CIK' + __cikValue__ + '/ifrs-full/' + __variName__ + '.json', headers = __requHead__)
    else:
        pass

In [8]:
def plotData(__tickSymb__, __tickTitl__, __dat1Dict__, __dat1Labe__, __dat2Dict__ = {}, __dat2Labe__ = "", __dualPlot__ = False):
    
    if __dualPlot__ and int(__dat1Dict__.status_code) == 200 and int(__dat2Dict__.status_code) == 200:
        # DERIVE CURRENCY UNIT
        currUnit = list(__dat1Dict__.json()["units"].keys())
        if len(currUnit) == 1:
            prefUnit = str(currUnit[0])
        else:
            for item in currUnit:
                if str(item) == "USD":
                    prefUnit = item
                    break
                else:
                    prefUnit = str(currUnit[0])
        # POST-PROCESS DATASET 1
        try:
            __dat1Dict__ = pd.json_normalize(__dat1Dict__.json()["units"][prefUnit])
            __dat1Dict__["filed"] = pd.to_datetime(__dat1Dict__["filed"])
            __dat1Dict__ = __dat1Dict__.sort_values("end")
        except Exception:
            try:
                __dat1Dict__ = pd.json_normalize(__dat1Dict__.json()["units"][prefUnit + "/shares"])
                __dat1Dict__["filed"] = pd.to_datetime(__dat1Dict__["filed"])
                __dat1Dict__ = __dat1Dict__.sort_values("end")
            except Exception:
                __dat1Dict__ = pd.json_normalize(__dat1Dict__.json()["units"]["shares"])
                __dat1Dict__["filed"] = pd.to_datetime(__dat1Dict__["filed"])
                __dat1Dict__ = __dat1Dict__.sort_values("end")
        # POST-PROCESS DATASET 2
        try:
            __dat2Dict__ = pd.json_normalize(__dat2Dict__.json()["units"][prefUnit])
            __dat2Dict__["filed"] = pd.to_datetime(__dat2Dict__["filed"])
            __dat2Dict__ = __dat2Dict__.sort_values("end")
        except Exception:
            try:
                __dat2Dict__ = pd.json_normalize(__dat2Dict__.json()["units"][prefUnit + "/shares"])
                __dat2Dict__["filed"] = pd.to_datetime(__dat2Dict__["filed"])
                __dat2Dict__ = __dat2Dict__.sort_values("end")
            except Exception:
                __dat2Dict__ = pd.json_normalize(__dat2Dict__.json()["units"]["shares"])
                __dat2Dict__["filed"] = pd.to_datetime(__dat2Dict__["filed"])
                __dat2Dict__ = __dat2Dict__.sort_values("end")
    
        fig = make_subplots(specs = [[{"secondary_y": True}]])
        
        fig.add_trace(go.Scatter(x = __dat1Dict__["end"], 
                                 y = __dat1Dict__["val"], 
                                 name = __dat1Labe__), 
                      secondary_y = False
                     )

        fig.add_trace(go.Scatter(x = __dat2Dict__["end"], 
                                 y = __dat2Dict__["val"], 
                                 name = __dat2Labe__),
                      secondary_y = True
                     )

        fig.update_layout(
            width = 990,
            height = 600,
            paper_bgcolor = 'White',
            plot_bgcolor = '#fafafa',
            hovermode = 'closest',
            title_text = "{} / {} Over Time for {} ({})".format(__dat1Labe__, __dat2Labe__, __tickSymb__, tickTitl),
            title_x = 0.46,
            xaxis = dict(title = "Time"),
            yaxis = dict(title = "{} / {} | Currency: {}".format(__dat1Labe__, __dat2Labe__, prefUnit)),
            showlegend = True,
            legend = dict(orientation = 'h', yanchor = 'bottom', y = 0.9, xanchor = 'left', x = 0.0)
        )
        
        fig.update_xaxes(showline = True, 
                         linewidth = 2, 
                         linecolor = 'Black',
                         showgrid = True,
                         gridwidth = 1,
                         gridcolor = 'LightGrey'
                        )
        fig.update_yaxes(showline = True, 
                         linewidth = 2, 
                         linecolor = 'Black',
                         showgrid = True,
                         gridwidth = 1,
                         gridcolor = 'LightGrey'
                        )
        
        fig.show()
        
    elif not __dualPlot__ and int(__dat1Dict__.status_code) == 200:
        # DERIVE CURRENCY UNIT
        currUnit = list(__dat1Dict__.json()["units"].keys())
        if len(currUnit) == 1:
            prefUnit = str(currUnit[0])
        else:
            for item in currUnit:
                if str(item) == "USD":
                    prefUnit = item
                    break
                else:
                    prefUnit = str(currUnit[0])
        # POST-PROCESS DATASET 1
        try:
            __dat1Dict__ = pd.json_normalize(__dat1Dict__.json()["units"][prefUnit])
            __dat1Dict__["filed"] = pd.to_datetime(__dat1Dict__["filed"])
            __dat1Dict__ = __dat1Dict__.sort_values("end")
        except Exception:
            try:
                __dat1Dict__ = pd.json_normalize(__dat1Dict__.json()["units"][prefUnit + "/shares"])
                __dat1Dict__["filed"] = pd.to_datetime(__dat1Dict__["filed"])
                __dat1Dict__ = __dat1Dict__.sort_values("end")
            except Exception:
                __dat1Dict__ = pd.json_normalize(__dat1Dict__.json()["units"]["shares"])
                __dat1Dict__["filed"] = pd.to_datetime(__dat1Dict__["filed"])
                __dat1Dict__ = __dat1Dict__.sort_values("end")
                
        fig = make_subplots(specs = [[{"secondary_y": True}]])
        
        fig.add_trace(go.Scatter(x = __dat1Dict__["end"], 
                                 y = __dat1Dict__["val"], 
                                 name = __dat1Labe__), 
                      secondary_y = False
                     )
        
        fig.update_layout(
            width = 990,
            height = 600,
            paper_bgcolor = 'White',
            plot_bgcolor = '#fafafa',
            hovermode = 'closest',
            title_text = "{} Over Time for {} ({})".format(__dat1Labe__, __tickSymb__, tickTitl),
            title_x = 0.46,
            xaxis = dict(title = "Time"),
            yaxis = dict(title = "{} | Currency: {}".format(__dat1Labe__, prefUnit)),
            showlegend = True,
            legend = dict(orientation = 'h', yanchor = 'bottom', y = 0.9, xanchor = 'left', x = 0.0)
        )
        
        fig.update_xaxes(showline = True, 
                         linewidth = 2, 
                         linecolor = 'Black',
                         showgrid = True,
                         gridwidth = 1,
                         gridcolor = 'LightGrey'
                        )
        fig.update_yaxes(showline = True, 
                         linewidth = 2, 
                         linecolor = 'Black',
                         showgrid = True,
                         gridwidth = 1,
                         gridcolor = 'LightGrey'
                        )
        
        fig.show()
    
    else:
        pass

In [9]:
# DERIVE DATAFRAMES
for indx in range(len(requResp)):
    if str(requResp.index[indx]) == __tickSymb:
        # GET RESPONSE TICKER TITLE
        tickTitl = str(requResp['title'].iloc[indx])
        # GET RESPONSE DATA
        #
        # 
        # == LIABILITIES
        liabl0DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'DebtInstrumentFairValue')
        sleep(0.1)
        #
        liabl1DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'Liabilities')
        if int(liabl1DF.status_code) != 200:
            liabl1DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                                   __accoType__ = 'ifrs', 
                                   __variName__ = 'Liabilities')
            sleep(0.1)
        else:
            sleep(0.1)
        #
        liabl2DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'LiabilitiesAndStockholdersEquity')
        if int(liabl2DF.status_code) != 200:
            liabl2DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                                   __accoType__ = 'ifrs', 
                                   __variName__ = 'EquityAndLiabilities')
            sleep(0.1)
        else:
            sleep(0.1)
        #
        liabl3DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'LiabilitiesCurrent')
        if int(liabl3DF.status_code) != 200:
            liabl3DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                                   __accoType__ = 'ifrs', 
                                   __variName__ = 'CurrentLiabilities')
            sleep(0.1)
        else:
            sleep(0.1)
        #
        liabl4DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'LiabilitiesFairValueDisclosure')
        sleep(0.1)
        #
        liabl5DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'LiabilityForUncertainTaxPositionsNoncurrent')
        sleep(0.1)
        #
        liabl6DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'LongTermDebt')
        if int(liabl6DF.status_code) != 200:
            liabl6DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                                   __accoType__ = 'ifrs', 
                                   __variName__ = 'LongtermBorrowings')
            sleep(0.1)
        else:
            sleep(0.1)
        #
        liabl7DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'LongTermDebtCurrent')
        sleep(0.1)
        #
        liabl8DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearFive')
        sleep(0.1)
        #
        liabl9DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'LongTermDebtNoncurrent')
        sleep(0.1)  
        #
        liab10DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'OtherLiabilitiesNoncurrent')
        if int(liab10DF.status_code) != 0:
            liab10DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                                   __accoType__ = 'ifrs', 
                                   __variName__ = 'NoncurrentLiabilities')
            sleep(0.1)
        else:
            sleep(0.1)
        #
        liab11DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'DeferredIncomeTaxLiabilities')
        sleep(0.1)
        #
        liab12DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'DeferredIncomeTaxLiabilitiesNet')
        sleep(0.1)
        #
        #
        #
        sleep(1.0)
        #
        #
        #
        # == ASSETS
        asset0DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'Assets')
        if int(asset0DF.status_code) != 200:
            asset0DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                                   __accoType__ = 'ifrs', 
                                   __variName__ = 'Assets')
            sleep(0.1)
        else:
            sleep(0.1)
        #
        asset1DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'AssetsCurrent')
        if int(asset1DF.status_code) != 200:
            asset1DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                                   __accoType__ = 'ifrs', 
                                   __variName__ = 'CurrentAssets')
            sleep(0.1)
        else:
            sleep(0.1)
        #
        asset2DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'AssetsFairValueDisclosure')
        sleep(0.1)
        #
        asset3DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'OtherAssetsCurrent')
        sleep(0.1)
        #
        asset4DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'OtherAssetsMiscellaneousNoncurrent')
        sleep(0.1)
        #
        asset5DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'OtherAssetsNoncurrent')
        sleep(0.1)
        #
        #
        #
        sleep(1.0)
        #
        #
        #
        # == PROFIT
        profi0DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'EarningsPerShareBasic')
        if int(profi0DF.status_code) != 200:
            profi0DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                                   __accoType__ = 'ifrs', 
                                   __variName__ = 'BasicEarningsLossPerShare')
            sleep(0.1)
        else:
            sleep(0.1)
        #
        profi1DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'EarningsPerShareBasicAndDiluted')
        sleep(0.1)
        #
        profi2DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'EarningsPerShareDiluted')
        if int(profi2DF.status_code) != 200:
            profi2DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                                   __accoType__ = 'ifrs', 
                                   __variName__ = 'DilutedEarningsLossPerShare')
            sleep(0.1)
        else:
            sleep(0.1)
        #
        profi3DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'RevenueFromContractWithCustomerExcludingAssessedTax')
        sleep(0.1)
        #
        profi4DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'RevenueRemainingPerformanceObligation')
        sleep(0.1)
        #
        profi5DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'Revenues')
        if int(profi5DF.status_code) != 200:
            profi5DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                                   __accoType__ = 'ifrs', 
                                   __variName__ = 'AccountingProfit')
            sleep(0.1)
        else:
            sleep(0.1)
        #
        profi6DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'GrossProfit')
        if int(profi6DF.status_code) != 200:
            profi6DF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                                   __accoType__ = 'ifrs', 
                                   __variName__ = 'ProfitLoss')
            sleep(0.1)
        else:
            sleep(0.1)
        #
        #
        #
        sleep(1.0)
        #
        #
        #
        # == OUTSTANDING STOCKS - GAAP AND IFRS
        outStoDF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                               __accoType__ = 'gaap', 
                               __variName__ = 'CommonStockSharesOutstanding')
        if int(outStoDF.status_code) != 200:
            outStoDF = requestData(__cikValue__ = str(requResp['cik_str'].iloc[indx]), 
                                   __accoType__ = 'ifrs', 
                                   __variName__ = 'NumberOfSharesOutstanding')
            sleep(0.1)
        else:
            sleep(0.1)
        
        break

# SEC/Liabilities

In [10]:
# PLOT DATA:
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = liabl0DF,
         __dat1Labe__ = 'DebtInstrumentFairValue'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = liabl1DF,
         __dat1Labe__ = 'Liabilities'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = liabl2DF,
         __dat1Labe__ = 'LiabilitiesAndStockholdersEquity'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = liabl3DF,
         __dat1Labe__ = 'LiabilitiesCurrent'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = liabl4DF,
         __dat1Labe__ = 'LiabilitiesFairValueDisclosure'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = liabl5DF,
         __dat1Labe__ = 'LiabilityForUncertainTaxPositionsNoncurrent'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = liabl6DF,
         __dat1Labe__ = 'LongTermDebt'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = liabl7DF,
         __dat1Labe__ = 'LongTermDebtCurrent'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = liabl8DF,
         __dat1Labe__ = 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearFive'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = liabl9DF,
         __dat1Labe__ = 'LongTermDebtNoncurrent'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = liab10DF,
         __dat1Labe__ = 'OtherLiabilitiesNoncurrent'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = liab11DF,
         __dat1Labe__ = 'DeferredIncomeTaxLiabilities'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = liab12DF,
         __dat1Labe__ = 'DeferredIncomeTaxLiabilitiesNet'
        )

# SEC/Assets

In [11]:
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = asset0DF,
         __dat1Labe__ = 'Assets'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = asset1DF,
         __dat1Labe__ = 'AssetsCurrent'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = asset2DF,
         __dat1Labe__ = 'AssetsFairValueDisclosure'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = asset3DF,
         __dat1Labe__ = 'OtherAssetsCurrent'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = asset4DF,
         __dat1Labe__ = 'OtherAssetsMiscellaneousNoncurrent'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = asset5DF,
         __dat1Labe__ = 'OtherAssetsNoncurrent'
        )

# SEC/Earnings/Revenue/Profit

In [12]:
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = profi0DF,
         __dat1Labe__ = 'EarningsPerShareBasic'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = profi1DF,
         __dat1Labe__ = 'EarningsPerShareBasicAndDiluted'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = profi2DF,
         __dat1Labe__ = 'EarningsPerShareDiluted'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = profi3DF,
         __dat1Labe__ = 'RevenueFromContractWithCustomerExcludingAssessedTax'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = profi4DF,
         __dat1Labe__ = 'RevenueRemainingPerformanceObligation'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = profi5DF,
         __dat1Labe__ = 'Revenues'
        )
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = profi6DF,
         __dat1Labe__ = 'GrossProfit'
        )

# SEC/Shares Outstanding

In [13]:
plotData(__tickSymb__ = __tickSymb, 
         __tickTitl__ = tickTitl,
         __dat1Dict__ = outStoDF,
         __dat1Labe__ = 'CommonStockSharesOutstanding'
        )

## References 

In [14]:
# [1] https://www.sec.gov/edgar/sec-api-documentation
# [2] https://www.sec.gov/edgar/search/
# [3] https://www.sec.gov/oiea/Article/edgarguide.html
# [4] https://www.investopedia.com/ask/answers/011315/what-difference-between-gaap-and-ifrs.asp